In [0]:
# ─────────────────────────────────────────────────────────────
# GOLD — fact_ball  (grain: 1 row per delivery; the star fact)
# deliveries + ball_phase + dim_match keys, resolved ids carried through
# ─────────────────────────────────────────────────────────────
from pyspark.sql import functions as F

CATALOG = "cricket"
d  = spark.table(f"{CATALOG}.silver.deliveries")
bp = spark.table(f"{CATALOG}.silver.ball_phase")
dm = spark.table(f"{CATALOG}.silver.dim_match").select(
        "match_id", "start_date", "event_name", "season", "venue")

# match-level keys (recompute the same hashes the dims used, so FKs line up)
mk = (dm
    .withColumn("date_key",   F.date_format("start_date", "yyyyMMdd").cast("int"))
    .withColumn("series_key", F.xxhash64(F.concat_ws("|",
                    F.coalesce("event_name", F.lit("(no event)")), "season")))
    .withColumn("venue_key",  F.xxhash64(F.lower(F.trim("venue"))))
    .select("match_id", "date_key", "series_key", "venue_key"))

fact = (d
    # phase per ball
    .join(bp, ["match_id", "innings_number", "over_number", "ball_seq"], "left")
    # match-level FKs
    .join(mk, "match_id", "left")
    # team keys (same normalisation as dim_team)
    .withColumn("batting_team_key", F.xxhash64(F.lower(F.trim(F.col("batting_team")))))
    .withColumn("bowling_team_key", F.xxhash64(F.lower(F.trim(F.col("bowling_team")))))
    # derived measures
    .withColumn("is_legal_ball",
        (F.col("extra_wides") == 0) & (F.col("extra_noballs") == 0))
    .withColumn("is_bowler_wicket",
        F.col("is_wicket") &
        F.col("dismissal_kind").isin("bowled","caught","caught and bowled",
                                     "lbw","stumped","hit wicket"))
)

In [0]:
# select the final star-shaped column set
fact_ball = fact.select(
    # degenerate dims (on the fact)
    "match_id", "innings_number", "over_number", "ball_seq",
    "is_super_over",
    # foreign keys
    "date_key", "series_key", "venue_key",
    "batting_team_key", "bowling_team_key",
    F.col("batter_id"), F.col("bowler_id"),
    F.col("non_striker_id"), F.col("player_out_id"),
    "phase_key",
    # measures
    "runs_batter", "runs_extras", "runs_total",
    "extra_wides", "extra_noballs", "extra_byes", "extra_legbyes", "extra_penalty",
    "is_legal_ball", "is_wicket", "is_bowler_wicket", "wicket_count",
    "non_boundary",
)

fact_ball.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .clusterBy("match_id").saveAsTable(f"{CATALOG}.gold.fact_ball")

In [0]:
# ─────────────────────────────────────────────────────────────
# Verify — grain, FK integrity, and a real stat computed off the fact
# ─────────────────────────────────────────────────────────────
fb = spark.table(f"{CATALOG}.gold.fact_ball")

# 1) grain preserved: fact_ball == deliveries
print("fact_ball:", fb.count(), "| deliveries:", d.count())

# 2) FK integrity: no orphan keys against the dims
print("orphan date_key:",
      fb.join(spark.table(f"{CATALOG}.silver.dim_calendar"), "date_key", "left")
        .where("date is null").count())
print("null batter_id on legal balls:",
      fb.where("is_legal_ball and batter_id is null").count())

# 3) a real stat: top run-scorers (batting), joined to dim_player
spark.sql(f"""
  SELECT p.canonical_name,
         sum(f.runs_batter)                              AS runs,
         sum(CASE WHEN f.is_legal_ball THEN 1 ELSE 0 END) AS balls_faced,
         round(100.0*sum(f.runs_batter)/
               nullif(sum(CASE WHEN f.is_legal_ball THEN 1 ELSE 0 END),0),1) AS strike_rate
  FROM {CATALOG}.gold.fact_ball f
  JOIN {CATALOG}.silver.dim_player p ON f.batter_id = p.person_id
  WHERE NOT f.is_super_over
  GROUP BY p.canonical_name
  ORDER BY runs DESC LIMIT 10
""").show(truncate=False)